In [1]:
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [2]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [17]:
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "rms_energy",
    frame_size = 128,
    hop_length = 32
)


print(data.get_samples().shape)
print(data.get_labels().shape)

(3309, 1126)
(3309,)


In [18]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.7799352750809061


In [19]:
torch.manual_seed(42)

In [20]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/zcr_1.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="./model_paths/zcr_1.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.578448, Train accuracy: 0.6697
[INFO] EPOCH: 2/20
Train loss: 0.513334, Train accuracy: 0.7297
[INFO] EPOCH: 3/20
Train loss: 0.496725, Train accuracy: 0.7390
[INFO] EPOCH: 4/20
Train loss: 0.485874, Train accuracy: 0.7507
[INFO] EPOCH: 5/20
Train loss: 0.477950, Train accuracy: 0.7567
[INFO] EPOCH: 6/20
Train loss: 0.474480, Train accuracy: 0.7740
[INFO] EPOCH: 7/20
Train loss: 0.469899, Train accuracy: 0.7693
[INFO] EPOCH: 8/20
Train loss: 0.464076, Train accuracy: 0.7720
[INFO] EPOCH: 9/20
Train loss: 0.460771, Train accuracy: 0.7737
[INFO] EPOCH: 10/20
Train loss: 0.454579, Train accuracy: 0.7703
[INFO] EPOCH: 11/20
Train loss: 0.446201, Train accuracy: 0.7863
[INFO] EPOCH: 12/20
Train loss: 0.444581, Train accuracy: 0.7853
[INFO] EPOCH: 13/20
Train loss: 0.441041, Train accuracy: 0.7873
[INFO] EPOCH: 14/20
Train loss: 0.439183, Train accuracy: 0.7820
[INFO] EPOCH: 15/20
Train loss: 0.438794, Train accuracy: 0.7767
[INFO] EPOCH: 16/20
Train loss: 0.

In [21]:
torch.manual_seed(42)

In [22]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_2_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/zcr_2.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="./model_paths/zcr_2.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.559835, Train accuracy: 0.6833
[INFO] EPOCH: 2/20
Train loss: 0.497494, Train accuracy: 0.7390
[INFO] EPOCH: 3/20
Train loss: 0.482955, Train accuracy: 0.7537
[INFO] EPOCH: 4/20
Train loss: 0.464134, Train accuracy: 0.7693
[INFO] EPOCH: 5/20
Train loss: 0.452864, Train accuracy: 0.7783
[INFO] EPOCH: 6/20
Train loss: 0.444616, Train accuracy: 0.7817
[INFO] EPOCH: 7/20
Train loss: 0.439172, Train accuracy: 0.7860
[INFO] EPOCH: 8/20
Train loss: 0.426352, Train accuracy: 0.7973
[INFO] EPOCH: 9/20
Train loss: 0.421758, Train accuracy: 0.7983
[INFO] EPOCH: 10/20
Train loss: 0.425759, Train accuracy: 0.7903
[INFO] EPOCH: 11/20
Train loss: 0.413916, Train accuracy: 0.7997
[INFO] EPOCH: 12/20
Train loss: 0.403365, Train accuracy: 0.8060
[INFO] EPOCH: 13/20
Train loss: 0.396983, Train accuracy: 0.8107
[INFO] EPOCH: 14/20
Train loss: 0.391126, Train accuracy: 0.8080
[INFO] EPOCH: 15/20
Train loss: 0.386107, Train accuracy: 0.8113
[INFO] EPOCH: 16/20
Train loss: 0.

In [23]:
torch.manual_seed(42)

In [24]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_3_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/zcr_3.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=30)

loops.test(model_path="./model_paths/zcr_3.pth", test_loader=test_loader)

[INFO] EPOCH: 1/30
Train loss: 0.560068, Train accuracy: 0.6733
[INFO] EPOCH: 2/30
Train loss: 0.504402, Train accuracy: 0.7367
[INFO] EPOCH: 3/30
Train loss: 0.475295, Train accuracy: 0.7593
[INFO] EPOCH: 4/30
Train loss: 0.470211, Train accuracy: 0.7627
[INFO] EPOCH: 5/30
Train loss: 0.451556, Train accuracy: 0.7867
[INFO] EPOCH: 6/30
Train loss: 0.457505, Train accuracy: 0.7770
[INFO] EPOCH: 7/30
Train loss: 0.454407, Train accuracy: 0.7783
[INFO] EPOCH: 8/30
Train loss: 0.434489, Train accuracy: 0.7923
[INFO] EPOCH: 9/30
Train loss: 0.429332, Train accuracy: 0.7907
[INFO] EPOCH: 10/30
Train loss: 0.421862, Train accuracy: 0.7947
[INFO] EPOCH: 11/30
Train loss: 0.416843, Train accuracy: 0.7963
[INFO] EPOCH: 12/30
Train loss: 0.404610, Train accuracy: 0.8137
[INFO] EPOCH: 13/30
Train loss: 0.402970, Train accuracy: 0.8010
[INFO] EPOCH: 14/30
Train loss: 0.390980, Train accuracy: 0.8053
[INFO] EPOCH: 15/30
Train loss: 0.385480, Train accuracy: 0.8110
[INFO] EPOCH: 16/30
Train loss: 0.